<h1 style = "color : deeppink">Column Transformer

In [36]:
import pandas as pd 
import numpy as np 

In [37]:
df = pd.read_csv('customer.csv')

In [38]:
df.sample(5)   # gender is a nominal column and review and education is the ordinal columnn

,age,gender,review,education,purchased
28,48,Male,Poor,School,No
46,64,Female,Poor,PG,No
19,97,Male,Poor,PG,Yes
36,34,Female,Good,UG,Yes
21,32,Male,Average,PG,No


<h1 style = "color : deeppink">Normal Way</h1>
without using transformers we will do nominal encoding for the gender column and ordinal encoding for the review and education column

In [39]:
# splitting the dataset into train test 

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 0:4],df.iloc[:, -1], test_size = 0.2, random_state = 2)
X_train.shape

(40, 4)

In [40]:
X_train.head(6)

,age,gender,review,education
24,16,Female,Average,PG
48,39,Female,Good,UG
17,22,Female,Poor,UG
12,51,Male,Poor,School
27,69,Female,Poor,PG
33,89,Female,Good,PG


In [41]:
# one hot encoding 
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(drop = 'first', dtype = int, sparse_output = False)

# fitting and transforming the training data 
X_train_gender = ohe.fit_transform(X_train[['gender']])

# also for the test data 
X_test_gender = ohe.fit_transform(X_test[['gender']])

X_train_gender.shape

(40, 1)

In [42]:
# ordinal encoding for the review and education column 

from sklearn.preprocessing import OrdinalEncoder
oe = OrdinalEncoder(categories = [['Poor', 'Average', 'Good'], ['School', 'UG', 'PG']], dtype = int)

# fit and transform the train data 
X_train_review_educaiton = oe.fit_transform(X_train[['review', 'education']])

# also for the test data 
X_test_review_education = oe.fit_transform(X_test[['review', 'education']])

X_train_review_educaiton[0:10]    # the ordinal encoder class returns an array so we can access the elements of the fitted data

array([[1, 2],
       [2, 1],
       [0, 1],
       [0, 0],
       [0, 2],
       [2, 2],
       [0, 1],
       [2, 2],
       [2, 0],
       [0, 2]])

In [43]:
# Extracting the numerical column age 
X_train_age = X_train.drop(columns = ['review', 'education', 'gender']).values

X_test_age = X_test.drop(columns = ['review', 'education', 'gender']).values

X_train_age.shape

(40, 1)

In [44]:
# concatenating all the transforms in the single dataset 

X_train_transformed = np.concatenate((X_train_age, X_train_gender, X_train_review_educaiton), axis = 1)

X_test_transformed = np.concatenate((X_test_age, X_test_gender, X_test_review_education), axis = 1)

X_train_transformed.shape

(40, 4)

In [46]:
X_train_transformed[1:10]

array([[39,  0,  2,  1],
       [22,  0,  0,  1],
       [51,  1,  0,  0],
       [69,  0,  0,  2],
       [89,  0,  2,  2],
       [59,  1,  0,  1],
       [70,  0,  2,  2],
       [57,  0,  2,  0],
       [15,  1,  0,  2]])

<h1 style = "color : deeppink">Using Transformer library</h1>

In [47]:
from sklearn.compose import ColumnTransformer

In [48]:
col_transformer = ColumnTransformer(transformers = [
    ('tnf1', OneHotEncoder(sparse_output = False, drop = 'first'), ['gender']),
    ('tnf2', OrdinalEncoder(categories = [['Poor', 'Average', 'Good'], ['School', 'UG', 'PG']], dtype = int), ['review', 'education'])
], remainder = 'passthrough')

In [53]:
# fit the train data to the transformer 
col_transformer.fit_transform(X_train).shape

(40, 4)

In [54]:
# fit the test data to the transformer
col_transformer.fit_transform(X_test).shape

(10, 4)